# recon_emb sanity check — optimal embeddings

Test the **optimal** reconstruction runs saved by `experiments/recon_emb.py` (small arch:
encoder `(512,256)` / decoder `(256,512)` for the AE, `(512,512)` for the pretrained
decoders, dropout 0.1) under `recon_weights/*_opt/`, each saved as a self-contained
`ReconDecoder.pkl`.

Uses `recon_weights/sciplex3_with_emb.h5ad` (log-norm HVG `X`, the foundation-model
embeddings, and the learned `AE_128_opt` embedding in `obsm`).

Two checks:
1. **Embedding structure** — UMAP of the held-out **test** cells on each embedding
   (`AE_128_opt` + the FM embeddings), coloured by `cell_type` / `cell_line`.
2. **ReconDecoder reproduces the log** — load each `*_opt/ReconDecoder.pkl`, `decode` the test
   split, recompute `pure_reconstruction`, and confirm it matches the run's `metadata.json`
   metrics — verifying the unified save/load + decode path.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import jax
import matplotlib.pyplot as plt
from IPython.display import display

EXPERIMENTS_DIR = "/lustre/groups/ml01/workspace/xiaotong.fu/pancellflow/sub/experiments"
sys.path.insert(0, EXPERIMENTS_DIR)
import recon_metrics as rm
from recon_emb import assign_splits
from scaleflow.model._recon import ReconDecoder

sc.settings.verbosity = 1
print("jax devices:", jax.devices())

In [ ]:
# ---- configuration ----
EMB_H5AD = "/lustre/groups/ml01/workspace/xiaotong.fu/pancellflow/recon_weights/sciplex3_with_emb.h5ad"
WEIGHTS_DIR = Path("/lustre/groups/ml01/workspace/xiaotong.fu/pancellflow/recon_weights")
SPLIT_BY, SPLIT_RATIOS, SPLIT_RS = ["drug"], [0.7, 0.2, 0.1], 42
SPLITS = ("train", "val", "test")
UMAP_N = 20_000
SEED = 0
# embeddings to UMAP: the optimal learned AE latent + the foundation-model embeddings
UMAP_KEYS = ["AE_128_opt", "X_state", "X_scgpt", "X_scimilarity", "X_scconcept"]

In [ ]:
# ---- load embedded data ----
adata = sc.read_h5ad(EMB_H5AD)
X = np.asarray(adata.X.todense() if hasattr(adata.X, "todense") else adata.X, dtype=np.float32)
cell_type = np.asarray(adata.obs["cell_type"].values).astype(str)
cell_line = np.asarray(adata.obs["cell_line"].values).astype(str)
print("adata:", adata.shape)
print("obsm:", {k: adata.obsm[k].shape for k in adata.obsm})

In [ ]:
# ---- reproduce the held-out-drug split ----
splits = assign_splits(adata, SPLIT_BY, SPLIT_RATIOS, SPLIT_RS)
idx = {s: np.where(splits == s)[0] for s in SPLITS}
print("split sizes:", {s: len(idx[s]) for s in SPLITS})

## 1. Embedding structure — test-set UMAP by cell type

UMAP of the held-out test cells using each embedding directly, coloured by `cell_type` and
`cell_line`. The optimal AE latent (`AE_128_opt`) now reconstructs well, so it should show
clear structure.

In [ ]:
te = idx["test"]
sub = np.random.default_rng(SEED).choice(te, size=min(UMAP_N, len(te)), replace=False)
for key in [k for k in UMAP_KEYS if k in adata.obsm]:
    aj = ad.AnnData(
        np.asarray(adata.obsm[key][sub], dtype=np.float32),
        obs={"cell_type": cell_type[sub], "cell_line": cell_line[sub]},
    )
    sc.pp.neighbors(aj, use_rep="X")   # neighbours on the embedding itself
    sc.tl.umap(aj)
    sc.pl.umap(aj, color=["cell_type", "cell_line"],
               title=[f"{key} — cell_type", f"{key} — cell_line"],
               frameon=False, ncols=2, show=True)

## 2. ReconDecoder reproduces the log

Load every `*_opt/ReconDecoder.pkl`, `decode` the test split (feeding the embedding each
decoder expects, via `metadata.input_key`), recompute `pure_reconstruction`, and compare to
the saved `metadata.json` metrics. Diffs should be ~0.

In [ ]:
opt_runs = sorted(p for p in WEIGHTS_DIR.iterdir()
                  if (p / "ReconDecoder.pkl").exists() and p.name.endswith("_opt"))
print("optimal runs:", [p.name for p in opt_runs])

rows = []
for run_dir in opt_runs:
    rd = ReconDecoder.load(str(run_dir))            # unified self-contained load
    latent_te = np.asarray(adata.obsm[rd.input_key][idx["test"]], dtype=np.float32)
    pred = rd.decode(latent_te)                     # latent -> genes
    recomputed = rm.pure_reconstruction(X[idx["test"]], pred)
    logged = rd.metadata["metrics"]["test"]
    for k in recomputed:
        rows.append(dict(run=run_dir.name, input=rd.input_key, metric=k,
                         logged=float(logged[k]), recomputed=float(recomputed[k]),
                         abs_diff=abs(float(logged[k]) - float(recomputed[k]))))
    print(f"  {run_dir.name:<30} decode {tuple(latent_te.shape)} -> {tuple(pred.shape)}")

df = pd.DataFrame(rows)
max_diff = df["abs_diff"].max()
print(f"\nmax |logged - recomputed| (ReconDecoder.decode vs metadata) = {max_diff:.2e}")
print("CHECKOUT:", "OK" if max_diff < 1e-3 else "MISMATCH")

In [ ]:
# optimal-setting reconstruction ranking (test), from the loaded ReconDecoders:
tab = (df[df.metric == "reconstruction_r2"][["run", "input", "recomputed"]]
       .rename(columns={"recomputed": "recon_R2"}).sort_values("recon_R2", ascending=False)
       .reset_index(drop=True))
display(tab.round(3))

## 3. Perturbation-delta R² per reconstructed unit (test)

Control-subtracted drug effect: per `(cell_line, drug)`, `r2_score` / Pearson of
`(mean_treated − control)` true vs reconstructed. This isolates how well each unit preserves
the *perturbation effect*, beyond baseline expression.

In [ ]:
drug = np.asarray(adata.obs["drug"].values).astype(str)
is_ctrl = np.asarray(adata.obs["control"].values).astype(bool)
ctrl_mean = rm.control_means(X, cell_line, is_ctrl)   # baseline = each line's control-cell mean (true)
te = idx["test"]
labels_te, drug_te, treated_te = cell_line[te], drug[te], ~is_ctrl[te]

drows = []
for run_dir in opt_runs:
    rd = ReconDecoder.load(str(run_dir))
    pred = rd.decode(np.asarray(adata.obsm[rd.input_key][te], dtype=np.float32))
    dm = rm.delta_metrics(X[te], pred, labels_te, drug_te, treated_te, ctrl_mean, min_cells=20)
    drows.append(dict(unit=rd.input_key, **dm))
ddf = pd.DataFrame(drows).sort_values("r2_median", ascending=False).reset_index(drop=True)
display(ddf.round(3))

## Interpretation

- **Metric check:** `max |logged - recomputed| ~ 0` confirms the `ReconDecoder.pkl` round-trips
  (load → decode) and reproduces the training-time metrics — the unified save format is sound.
- **Ranking:** the learned `AE_128_opt` reconstructs best (its latent is trained for it), then
  `X_state` is the strongest fixed foundation-model embedding, well ahead of scGPT / scConcept /
  scimilarity.
- For decoding **flow** predictions (which live in `X_state`), `dec_X_state_opt` is the decoder
  to attach — the AE latent is not what the flow outputs.